<a href="https://colab.research.google.com/github/srilakshmi005/careai-healthcare-ai/blob/main/CareAI_ML_Risk_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("CareAI project started successfully!")
import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("All ML libraries are ready! ✅")
!pip -q install ucimlrepo

from ucimlrepo import fetch_ucirepo

# Load the UCI healthcare dataset
dataset = fetch_ucirepo(id=296)

X = dataset.data.features
y = dataset.data.targets

print("Dataset loaded successfully! ✅")
print("Features shape:", X.shape)
print("Target shape:", y.shape)
# Step 5: Inspect the healthcare dataset

print("===== DATASET INFORMATION =====")
print("Rows:", X.shape[0])
print("Features:", X.shape[1])

print("\n===== FEATURE NAMES =====")
print(X.columns.tolist())

print("\n===== FIRST 5 ROWS =====")
display(X.head())

print("\n===== TARGET VALUES =====")
print(y.head())

print("\n===== MISSING VALUES =====")
missing = X.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

print("\n===== TARGET DISTRIBUTION =====")
print(y.iloc[:, 0].value_counts())

leakage_keywords = ["readmitted", "discharge", "outcome", "death", "expired"]
print([c for c in X.columns if any(k in c.lower() for k in leakage_keywords)])
missing_report = pd.DataFrame({
    "missing_count": X.isnull().sum(),
    "missing_percent": (X.isnull().sum() / len(X) * 100).round(2)
})

print("===== MISSING DATA REPORT =====")
print(missing_report[missing_report["missing_count"] > 0].sort_values("missing_percent", ascending=False))
# Step 7: Create binary target
target = y.iloc[:, 0]

y_binary = (target == "<30").astype(int)

print("===== BINARY TARGET =====")
print(y_binary.value_counts())

print("\n0 = Not readmitted within 30 days")
print("1 = Readmitted within 30 days")
# Step 8: Prepare features


print("Original features:", X.shape[1])
print("Features after removing identifiers:", X_clean.shape[1])

print("\nRemaining features:")
print(X_clean.columns.tolist())

# Step 9: Train-test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clean,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print("===== TRAIN / TEST SPLIT =====")
print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining positive rate:", round(y_train.mean() * 100, 2), "%")
print("Testing positive rate:", round(y_test.mean() * 100, 2), "%")

# Step 10: Build preprocessing pipeline

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify column types
numeric_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

# Numeric preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine both
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

print("\nPreprocessing pipeline created successfully! ✅")

# Step 11: Train baseline Logistic Regression model

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

print("Training model...")

model.fit(X_train, y_train)

print("Model training completed successfully! ✅")

# Predictions
y_pred = model.predict(X_test)
y_probability = model.predict_proba(X_test)[:, 1]

print("\n===== MODEL EVALUATION =====")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("ROC-AUC:", round(roc_auc_score(y_test, y_probability), 4))

# Step 12: Train Random Forest model

from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training Random Forest...")

rf_model.fit(X_train, y_train)

print("Random Forest training completed! ✅")

rf_pred = rf_model.predict(X_test)
rf_probability = rf_model.predict_proba(X_test)[:, 1]

print("\n===== RANDOM FOREST EVALUATION =====")
print(classification_report(y_test, rf_pred))

print("ROC-AUC:", round(roc_auc_score(y_test, rf_probability), 4))

# Step 13: Train XGBoost model

!pip -q install xgboost

from xgboost import XGBClassifier

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training XGBoost...")

xgb_model.fit(X_train, y_train)

print("XGBoost training completed! ✅")

xgb_pred = xgb_model.predict(X_test)
xgb_probability = xgb_model.predict_proba(X_test)[:, 1]

print("\n===== XGBOOST EVALUATION =====")
print(classification_report(y_test, xgb_pred))

print("ROC-AUC:", round(roc_auc_score(y_test, xgb_probability), 4))

# Step 14: Threshold analysis for XGBoost

from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("===== THRESHOLD ANALYSIS =====")

for threshold in thresholds:
    threshold_pred = (xgb_probability >= threshold).astype(int)

    precision = precision_score(y_test, threshold_pred, zero_division=0)
    recall = recall_score(y_test, threshold_pred, zero_division=0)
    f1 = f1_score(y_test, threshold_pred, zero_division=0)

    print(
        "Threshold:", threshold,
        "| Precision:", round(precision, 3),
        "| Recall:", round(recall, 3),
        "| F1:", round(f1, 3)
    )

# Step 15: Final threshold-based evaluation

selected_threshold = 0.15

final_probability = xgb_model.predict_proba(X_test)[:, 1]
final_prediction = (final_probability >= selected_threshold).astype(int)

print("===== FINAL MODEL EVALUATION =====")
print("Selected threshold:", selected_threshold)

print("\nClassification Report:")
print(classification_report(
    y_test,
    final_prediction,
    target_names=["Not readmitted", "Readmitted within 30 days"]
))

print(
    "ROC-AUC:",
    round(roc_auc_score(y_test, final_probability), 4)
)

# Step 16: Model explainability using permutation importance

from sklearn.inspection import permutation_importance

print("Calculating feature importance...")

importance = permutation_importance(
    xgb_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std
})

importance_df = importance_df.sort_values(
    "importance_mean",
    ascending=False
)

print("\n===== TOP 15 FEATURES =====")
print(importance_df.head(15).to_string(index=False))

# Step 17: Remove potential leakage feature

X_production = X_clean.drop(
    columns=["discharge_disposition_id"],
    errors="ignore"
)

print("Original features:", X_clean.shape[1])
print("Production features:", X_production.shape[1])
print("Leakage feature removed: discharge_disposition_id")

# Step 18: Fresh train-test split without leakage feature

X_train_prod, X_test_prod, y_train_prod, y_test_prod = train_test_split(
    X_production,
    y_binary,
    test_size=0.20,
    random_state=42,
    stratify=y_binary
)

print("===== PRODUCTION TRAIN / TEST SPLIT =====")
print("Training samples:", X_train_prod.shape[0])
print("Testing samples:", X_test_prod.shape[0])

print("\nTraining positive rate:", round(y_train_prod.mean() * 100, 2), "%")
print("Testing positive rate:", round(y_test_prod.mean() * 100, 2), "%")

# Step 19: Production preprocessing pipeline

numeric_features_prod = X_train_prod.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features_prod = X_train_prod.select_dtypes(
    include=["object"]
).columns.tolist()

print("Numeric features:", len(numeric_features_prod))
print("Categorical features:", len(categorical_features_prod))

numeric_pipeline_prod = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline_prod = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_prod = ColumnTransformer([
    ("numeric", numeric_pipeline_prod, numeric_features_prod),
    ("categorical", categorical_pipeline_prod, categorical_features_prod)
])

print("\nProduction preprocessing pipeline created! ✅")

# Step 20: Train production XGBoost model

production_xgb = Pipeline([
    ("preprocessor", preprocessor_prod),
    ("classifier", XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ))
])

print("Training production XGBoost model...")

production_xgb.fit(X_train_prod, y_train_prod)

print("Production XGBoost training completed! ✅")

production_probability = production_xgb.predict_proba(X_test_prod)[:, 1]

print("\n===== PRODUCTION MODEL =====")
print("ROC-AUC:", round(
    roc_auc_score(y_test_prod, production_probability),4))

# Step 21: Production threshold analysis

production_thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("===== PRODUCTION THRESHOLD ANALYSIS =====")

for threshold in production_thresholds:
    pred = (production_probability >= threshold).astype(int)

    precision = precision_score(
        y_test_prod, pred, zero_division=0
    )

    recall = recall_score(
        y_test_prod, pred, zero_division=0
    )

    f1 = f1_score(
        y_test_prod, pred, zero_division=0
    )

    print(
        "Threshold:", threshold,
        "| Precision:", round(precision, 3),
        "| Recall:", round(recall, 3),
        "| F1:", round(f1, 3)
    )

# Step 22: Record model performance

model_results = pd.DataFrame({
    "model": [
        "Logistic Regression",
        "Random Forest",
        "XGBoost (potential leakage)",
        "XGBoost (production)"
    ],
    "roc_auc": [
        0.6440,
        0.6631,
        0.6920,
        0.6606
    ],
    "notes": [
        "Baseline model",
        "Improved baseline",
        "Included potentially unavailable discharge feature",
        "Leakage feature removed"
    ]
})

print("===== CAREAI MODEL COMPARISON =====")
display(model_results)

# Step 23: Save production model

import joblib
from datetime import datetime

MODEL_VERSION = "careai_xgboost_v1"

model_package = {
    "model": production_xgb,
    "threshold": 0.15,
    "model_version": MODEL_VERSION,
    "roc_auc": 0.6606,
    "features": X_production.columns.tolist(),
    "created_at": datetime.now().isoformat()
}

joblib.dump(model_package, "careai_model_v1.joblib")

print("===== MODEL SAVED =====")
print("Model version:", MODEL_VERSION)
print("File: careai_model_v1.joblib")
print("Threshold:", model_package["threshold"])
print("ROC-AUC:", model_package["roc_auc"])
print("Saved successfully! ✅")

# Step 24: Create model metadata

metadata = {
    "project": "CareAI",
    "model_version": "careai_xgboost_v1",
    "algorithm": "XGBoost",
    "prediction_target": "30-day readmission",
    "production_features": X_production.shape[1],
    "training_rows": X_train_prod.shape[0],
    "testing_rows": X_test_prod.shape[0],
    "roc_auc": 0.6606,
    "decision_threshold": 0.15,
    "leakage_feature_removed": "discharge_disposition_id"
}

print("===== CAREAI MODEL METADATA =====")

for key, value in metadata.items():
    print(key, ":", value)

# Step 25: Create CareAI SQL database

import sqlite3

connection = sqlite3.connect("careai_healthcare.db")

print("CareAI SQLite database created successfully! ✅")

# Step 26: Create healthcare table in SQLite

sql_data = X_production.copy()

# Add the prediction target
sql_data["readmitted_30d"] = y_binary.values

sql_data.to_sql(
    "patient_encounters",
    connection,
    if_exists="replace",
    index=False
)

print("Healthcare table created successfully! ✅")

cursor = connection.cursor()

cursor.execute(
    "SELECT COUNT(*) FROM patient_encounters"
)

row_count = cursor.fetchone()[0]

print("Rows stored in SQL:", row_count)

# Step 27: SQL analytics

query = """
SELECT
    readmitted_30d,
    COUNT(*) AS patient_count
FROM patient_encounters
GROUP BY readmitted_30d
ORDER BY readmitted_30d;
"""

sql_result = pd.read_sql_query(query, connection)

print("===== SQL READMISSION SUMMARY =====")
display(sql_result)

# Step 28: Healthcare risk summary using SQL

query = """
SELECT
    COUNT(*) AS total_encounters,
    SUM(readmitted_30d) AS readmitted_patients,
    ROUND(
        100.0 * SUM(readmitted_30d) / COUNT(*),
        2
    ) AS readmission_rate_percent
FROM patient_encounters;
"""

risk_summary = pd.read_sql_query(query, connection)

print("===== CAREAI SQL RISK SUMMARY =====")
display(risk_summary)

# Step 29: CareAI prediction function

def predict_readmission(patient_data):
    patient_df = pd.DataFrame([patient_data])

    probability = production_xgb.predict_proba(patient_df)[0, 1]

    threshold = 0.15

    prediction = int(probability >= threshold)

    if prediction == 1:
        risk = "HIGH RISK"
    else:
        risk = "LOW RISK"

    return {
        "prediction": prediction,
        "risk": risk,
        "probability": round(float(probability), 4),
        "threshold": threshold,
        "model_version": "careai_xgboost_v1"
    }

print("CareAI prediction function created successfully! ✅")

# Step 30: Test CareAI prediction

test_patient = X_test_prod.iloc[0].to_dict()

result = predict_readmission(test_patient)

print("===== CAREAI PREDICTION =====")
print("Prediction:", result["prediction"])
print("Risk:", result["risk"])
print("Probability:", result["probability"])
print("Threshold:", result["threshold"])
print("Model version:", result["model_version"])

# Step 31: Prediction logging

import json
from datetime import datetime

prediction_log = {
    "timestamp": datetime.now().isoformat(),
    "model_version": result["model_version"],
    "prediction": result["prediction"],
    "risk": result["risk"],
    "probability": result["probability"],
    "threshold": result["threshold"]
}

print("===== CAREAI PREDICTION LOG =====")
print(json.dumps(prediction_log, indent=2))

print("\nPrediction logged successfully! ✅")

# Step 32: Create prediction logs table

connection.execute("""
CREATE TABLE IF NOT EXISTS prediction_logs (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    model_version TEXT,
    prediction INTEGER,
    risk TEXT,
    probability REAL,
    threshold REAL
)
""")

connection.commit()

print("Prediction logs table created successfully! ✅")

# Step 33: Store prediction in SQL

connection.execute(
    """
    INSERT INTO prediction_logs
    (timestamp, model_version, prediction, risk, probability, threshold)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    (
        prediction_log["timestamp"],
        prediction_log["model_version"],
        prediction_log["prediction"],
        prediction_log["risk"],
        prediction_log["probability"],
        prediction_log["threshold"]
    )
)

connection.commit()

print("Prediction stored in SQL successfully! ✅")

# Verify prediction log

logs = pd.read_sql_query(
    "SELECT * FROM prediction_logs ORDER BY id DESC LIMIT 5",
    connection
)

print("\n===== RECENT CAREAI PREDICTIONS =====")
display(logs)

CareAI project started successfully!
Pandas: 2.2.2
NumPy: 2.0.2
Scikit-learn: 1.6.1
All ML libraries are ready! ✅


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Dataset loaded successfully! ✅
Features shape: (101766, 47)
Target shape: (101766, 1)
===== DATASET INFORMATION =====
Rows: 101766
Features: 47

===== FEATURE NAMES =====
['race', 'gender', 'age', 'weight', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'payer_code', 'medical_specialty', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3', 'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']

===== FIRST 5 ROWS =====


,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,...,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed
0,Caucasian,Female,[0-10),NaN,6,25,1,1,NaN,Pediatrics-Endocrinology,...,No,No,No,No,No,No,No,No,No,No
1,Caucasian,Female,[10-20),NaN,1,1,7,3,NaN,NaN,...,No,No,Up,No,No,No,No,No,Ch,Yes
2,AfricanAmerican,Female,[20-30),NaN,1,1,7,2,NaN,NaN,...,No,No,No,No,No,No,No,No,No,Yes
3,Caucasian,Male,[30-40),NaN,1,1,7,2,NaN,NaN,...,No,No,Up,No,No,No,No,No,Ch,Yes
4,Caucasian,Male,[40-50),NaN,1,1,7,1,NaN,NaN,...,No,No,Steady,No,No,No,No,No,Ch,Yes



===== TARGET VALUES =====
  readmitted
0         NO
1        >30
2         NO
3         NO
4         NO

===== MISSING VALUES =====
weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

===== TARGET DISTRIBUTION =====
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64
['discharge_disposition_id']
===== MISSING DATA REPORT =====
                   missing_count  missing_percent
weight                     98569            96.86
max_glu_serum              96420            94.75
A1Cresult                  84748            83.28
medical_specialty          49949            49.08
payer_code                 40256            39.56
race                        2273             2.23
diag_3                      1423             1.40
diag_2                       358         

...

,readmitted_30d,patient_count
0,0,90409
1,1,11357


===== CAREAI SQL RISK SUMMARY =====


,total_encounters,readmitted_patients,readmission_rate_percent
0,101766,11357,11.16


CareAI prediction function created successfully! ✅
===== CAREAI PREDICTION =====
Prediction: 0
Risk: LOW RISK
Probability: 0.1445
Threshold: 0.15
Model version: careai_xgboost_v1
===== CAREAI PREDICTION LOG =====
{
  "timestamp": "2026-08-18T12:39:01.810237",
  "model_version": "careai_xgboost_v1",
  "prediction": 0,
  "risk": "LOW RISK",
  "probability": 0.1445,
  "threshold": 0.15
}

Prediction logged successfully! ✅
Prediction logs table created successfully! ✅
Prediction stored in SQL successfully! ✅

===== RECENT CAREAI PREDICTIONS =====


,id,timestamp,model_version,prediction,risk,probability,threshold
0,2,2026-08-18T12:39:01.810237,careai_xgboost_v1,0,LOW RISK,0.1445,0.15
1,1,2026-08-18T11:47:34.410330,careai_xgboost_v1,0,LOW RISK,0.1445,0.15


In [ ]:
# Step 34: Verify SQL prediction logs

logs = pd.read_sql_query(
    "SELECT * FROM prediction_logs ORDER BY id DESC LIMIT 5",
    connection
)

print("===== RECENT CAREAI PREDICTIONS =====")
display(logs)
# Step 35: CareAI monitoring report

monitoring_query = """
SELECT
    COUNT(*) AS total_predictions,
    SUM(CASE WHEN prediction = 1 THEN 1 ELSE 0 END) AS high_risk_predictions,
    ROUND(AVG(probability), 4) AS average_probability,
    ROUND(MAX(probability), 4) AS maximum_probability
FROM prediction_logs;
"""

monitoring = pd.read_sql_query(
    monitoring_query,
    connection
)

print("===== CAREAI MONITORING REPORT =====")
display(monitoring)

# Step 36: Simple data drift monitoring

print("===== CAREAI DATA DRIFT CHECK =====")

numeric_drift = []

for column in numeric_features_prod:
    train_mean = X_train_prod[column].mean()
    test_mean = X_test_prod[column].mean()

    if train_mean != 0:
        change_percent = abs(
            (test_mean - train_mean) / train_mean
        ) * 100
    else:
        change_percent = 0

    numeric_drift.append({
        "feature": column,
        "train_mean": round(train_mean, 3),
        "test_mean": round(test_mean, 3),
        "change_percent": round(change_percent, 2)
    })

drift_report = pd.DataFrame(numeric_drift)

print("\n===== NUMERIC FEATURE DRIFT =====")
display(
    drift_report.sort_values(
        "change_percent",
        ascending=False
    )
)

# Step 37: Drift alert

DRIFT_THRESHOLD = 20

drift_report["drift_status"] = np.where(
    drift_report["change_percent"] >= DRIFT_THRESHOLD,
    "WARNING",
    "OK"
)

print("===== CAREAI DRIFT ALERT =====")

display(
    drift_report[
        ["feature", "change_percent", "drift_status"]
    ]
)

warnings = (drift_report["drift_status"] == "WARNING").sum()

print("\nFeatures requiring attention:", warnings)

if warnings > 0:
    print("⚠️ Drift warning detected")
else:
    print("✅ No significant drift detected")

# Step 38: Install FastAPI tools

!pip -q install fastapi uvicorn

print("FastAPI installed successfully! ✅")

# Step 39: Create CareAI FastAPI application

from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(
    title="CareAI",
    description="Healthcare 30-day readmission risk prediction API",
    version="1.0.0"
)

class PredictionRequest(BaseModel):
    patient_data: dict

@app.get("/")
def home():
    return {
        "service": "CareAI",
        "status": "running",
        "model_version": "careai_xgboost_v1"
    }

@app.post("/predict")
def predict(request: PredictionRequest):
    try:
        result = predict_readmission(request.patient_data)
        return result

    except Exception as e:
        return {
            "error": "Prediction failed",
            "message": str(e),
            "model_version": "careai_xgboost_v1"
        }
print("CareAI FastAPI application created successfully! ✅")

# Step 40: Test CareAI API

from fastapi.testclient import TestClient

client = TestClient(app)

response = client.get("/")

print("===== CAREAI API TEST =====")
print("Status code:", response.status_code)
print("Response:", response.json())

===== RECENT CAREAI PREDICTIONS =====


,id,timestamp,model_version,prediction,risk,probability,threshold
0,2,2026-08-18T12:39:01.810237,careai_xgboost_v1,0,LOW RISK,0.1445,0.15
1,1,2026-08-18T11:47:34.410330,careai_xgboost_v1,0,LOW RISK,0.1445,0.15


===== CAREAI MONITORING REPORT =====


,total_predictions,high_risk_predictions,average_probability,maximum_probability
0,2,0,0.1445,0.1445


===== CAREAI DATA DRIFT CHECK =====

===== NUMERIC FEATURE DRIFT =====


,feature,train_mean,test_mean,change_percent
7,number_emergency,0.200,0.189,5.70
6,number_outpatient,0.370,0.367,0.89
4,num_procedures,1.337,1.349,0.86
0,admission_type_id,2.021,2.035,0.71
1,admission_source_id,5.761,5.728,0.56
2,time_in_hospital,4.393,4.408,0.33
8,number_inpatient,0.635,0.636,0.17
3,num_lab_procedures,43.092,43.108,0.04
5,num_medications,16.021,16.025,0.03
9,number_diagnoses,7.423,7.422,0.00


===== CAREAI DRIFT ALERT =====


,feature,change_percent,drift_status
0,admission_type_id,0.71,OK
1,admission_source_id,0.56,OK
2,time_in_hospital,0.33,OK
3,num_lab_procedures,0.04,OK
4,num_procedures,0.86,OK
5,num_medications,0.03,OK
6,number_outpatient,0.89,OK
7,number_emergency,5.70,OK
8,number_inpatient,0.17,OK
9,number_diagnoses,0.00,OK



Features requiring attention: 0
✅ No significant drift detected
FastAPI installed successfully! ✅
CareAI FastAPI application created successfully! ✅
===== CAREAI API TEST =====
Status code: 200
Response: {'service': 'CareAI', 'status': 'running', 'model_version': 'careai_xgboost_v1'}


In [ ]:
# Step 41A: Test production model directly

one_patient = X_test_prod.iloc[[0]]

direct_result = production_xgb.predict_proba(one_patient)[0, 1]

print("Model probability:", float(direct_result))
print("Direct model test successful! ✅")

# Step 41B: JSON-safe API test

one_patient = X_test_prod.iloc[0]

safe_patient = {}

for column in X_test_prod.columns:
    value = one_patient[column]

    if pd.isna(value):
        safe_patient[column] = None
    elif hasattr(value, "item"):
        safe_patient[column] = value.item()
    else:
        safe_patient[column] = value

print("Patient fields:", len(safe_patient))

response = client.post(
    "/predict",
    json={"patient_data": safe_patient}
)

print("===== CAREAI PREDICTION API TEST =====")
print("Status code:", response.status_code)
print("Response:", response.json())

# Step 43: Test API error handling

bad_response = client.post(
    "/predict",
    json={"patient_data": {}}
)

print("===== INVALID REQUEST TEST =====")
print("Status code:", bad_response.status_code)
print("Response:", bad_response.json())

# Step 44: Automated API tests

print("===== CAREAI AUTOMATED TESTS =====")

# Test 1: Health endpoint
health_response = client.get("/")

assert health_response.status_code == 200
assert health_response.json()["service"] == "CareAI"

print("✅ Test 1 passed: Health endpoint")

# Test 2: Model prediction
one_patient = X_test_prod.iloc[0]

safe_patient = {}

for column in X_test_prod.columns:
    value = one_patient[column]

    if pd.isna(value):
        safe_patient[column] = None
    elif hasattr(value, "item"):
        safe_patient[column] = value.item()
    else:
        safe_patient[column] = value

prediction_response = client.post(
    "/predict",
    json={"patient_data": safe_patient}
)

assert prediction_response.status_code == 200

prediction_result = prediction_response.json()

assert "prediction" in prediction_result
assert "risk" in prediction_result
assert "probability" in prediction_result
assert "model_version" in prediction_result

print("✅ Test 2 passed: Prediction endpoint")

# Test 3: Invalid patient
bad_response = client.post(
    "/predict",
    json={"patient_data": {}}
)

assert bad_response.status_code == 200
assert "error" in bad_response.json()

print("✅ Test 3 passed: Error handling")

print("\n🎉 ALL CAREAI TESTS PASSED!")

# Step 45: Production model evaluation report

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Generate predictions using production threshold
production_predictions = (
    production_probability >= 0.15
).astype(int)

print("===== CAREAI PRODUCTION EVALUATION =====")

print("\nModel version:")
print("careai_xgboost_v1")

print("\nROC-AUC:")
print(round(
    roc_auc_score(y_test_prod, production_probability),
    4
))

print("\nDecision threshold:")
print("0.15")

print("\nClassification Report:")
print(
    classification_report(
        y_test_prod,
        production_predictions,
        target_names=[
            "Not readmitted",
            "Readmitted within 30 days"
        ],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test_prod,
        production_predictions
    )
)

print("\nLeakage feature removed:")
print("discharge_disposition_id")

print("\n===== EVALUATION COMPLETE =====")

# Step 46: Create requirements.txt

requirements = """
pandas
numpy
scikit-learn
xgboost
joblib
fastapi
uvicorn
pydantic
ucimlrepo
"""

with open("requirements.txt", "w") as file:
    file.write(requirements.strip())

print("requirements.txt created successfully! ✅")

# Step 47: Create CareAI README

readme = """
# 🏥 CareAI — Healthcare Readmission Risk Prediction

CareAI is an end-to-end machine learning system that predicts the risk of
30-day hospital readmission using de-identified healthcare encounter data.

## 🎯 Objective

Build a production-oriented ML pipeline that demonstrates:

- Data preprocessing
- Machine learning model development
- Data leakage detection
- Model evaluation
- Threshold optimization
- SQL data storage
- Prediction logging
- Monitoring
- Data drift detection
- FastAPI model serving
- Automated API testing
- Model versioning

## 📊 Dataset

The project uses the publicly available Diabetes 130-US Hospitals dataset
from the UCI Machine Learning Repository.

The dataset contains 101,766 healthcare encounters and 47 original features.

## 🤖 Models Evaluated

| Model | ROC-AUC |
|---|---:|
| Logistic Regression | 0.6440 |
| Random Forest | 0.6631 |
| XGBoost with potential leakage | 0.6920 |
| Production XGBoost | 0.6606 |

## 🔐 Data Leakage Handling

During feature analysis, `discharge_disposition_id` was identified as a
potentially unavailable feature at the intended prediction point.

It was removed from the production feature set.

The production model therefore uses 46 features.

## 🏆 Production Model

**Algorithm:** XGBoost

**Model version:** careai_xgboost_v1

**ROC-AUC:** 0.6606

**Decision threshold:** 0.15

The threshold was selected using development-set precision, recall and F1
analysis.

## 🗄️ SQL Data Layer

SQLite is used to store:

- Healthcare encounters
- Prediction logs

The prediction log records:

- Timestamp
- Model version
- Prediction
- Risk level
- Probability
- Decision threshold

## 🌐 API

CareAI provides a FastAPI service with:

### GET /

Health/status endpoint.

### POST /predict

Accepts patient information and returns:

- Prediction
- Risk level
- Probability
- Threshold
- Model version

## 📈 Monitoring

The project includes:

- Prediction monitoring
- Average prediction probability
- Maximum prediction probability
- Basic numeric feature drift monitoring
- Drift alerts

## 🧪 Testing

Automated tests cover:

- API health endpoint
- Prediction endpoint
- Invalid input/error handling

## 🛠️ Technology Stack

Python
Pandas
NumPy
Scikit-learn
XGBoost
SQLite
SQL
FastAPI
Pydantic
Joblib
UCI Machine Learning Repository

## ⚠️ Disclaimer

This is an educational/research project using publicly available
de-identified data. It is not a clinical decision-support system and
should not be used for real patient-care decisions.

## 🚀 Future Improvements

- Cloud deployment
- Docker containerization
- CI/CD
- MLflow experiment tracking
- Advanced statistical drift detection
- Fairness evaluation
- RAG-based healthcare information assistant
- LLM evaluation and guardrails
- Production observability
"""

with open("README.md", "w") as file:
    file.write(readme.strip())

print("README.md created successfully! ✅")

Model probability: 0.14445772767066956
Direct model test successful! ✅
Patient fields: 46
===== CAREAI PREDICTION API TEST =====
Status code: 200
Response: {'prediction': 0, 'risk': 'LOW RISK', 'probability': 0.1397, 'threshold': 0.15, 'model_version': 'careai_xgboost_v1'}
===== INVALID REQUEST TEST =====
Status code: 200
Response: {'error': 'Prediction failed', 'message': "'NoneType' object is not iterable", 'model_version': 'careai_xgboost_v1'}
===== CAREAI AUTOMATED TESTS =====
✅ Test 1 passed: Health endpoint
✅ Test 2 passed: Prediction endpoint
✅ Test 3 passed: Error handling

🎉 ALL CAREAI TESTS PASSED!
===== CAREAI PRODUCTION EVALUATION =====

Model version:
careai_xgboost_v1

ROC-AUC:
0.6606

Decision threshold:
0.15

Classification Report:
                           precision    recall  f1-score   support

           Not readmitted       0.91      0.84      0.87     18083
Readmitted within 30 days       0.21      0.35      0.26      2271

                 accuracy              

In [ ]:
# Step 48: Check CareAI project files

import os

files = [
    "README.md",
    "requirements.txt",
    "careai_model_v1.joblib",
    "careai_healthcare.db"
]

print("===== CAREAI PROJECT FILES =====")

for file in files:
    if os.path.exists(file):
        size = os.path.getsize(file)
        print("✅", file, "-", size, "bytes")
    else:
        print("❌", file, "- NOT FOUND")

===== CAREAI PROJECT FILES =====
✅ README.md - 2841 bytes
✅ requirements.txt - 75 bytes
✅ careai_model_v1.joblib - 873147 bytes
✅ careai_healthcare.db - 16343040 bytes


In [ ]:
# Step 49: Create GitHub-ready CareAI structure

import os

folders = [
    "CareAI",
    "CareAI/notebooks",
    "CareAI/app",
    "CareAI/tests",
    "CareAI/monitoring"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("===== CAREAI PROJECT STRUCTURE =====")

for folder in folders:
    print("✅", folder)

print("\nGitHub-ready project structure created! ✅")

===== CAREAI PROJECT STRUCTURE =====
✅ CareAI
✅ CareAI/notebooks
✅ CareAI/app
✅ CareAI/tests
✅ CareAI/monitoring

GitHub-ready project structure created! ✅


In [ ]:
# Step 50: Copy project documentation

import shutil
import os

shutil.copy("README.md", "CareAI/README.md")
shutil.copy("requirements.txt", "CareAI/requirements.txt")

print("===== CAREAI DOCUMENTATION =====")
print("✅ README.md copied")
print("✅ requirements.txt copied")
print("✅ Documentation ready for GitHub!")

===== CAREAI DOCUMENTATION =====
✅ README.md copied
✅ requirements.txt copied
✅ Documentation ready for GitHub!
